In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Flatten, Dense, Dropout
import matplotlib.pyplot as plt

In [2]:
imdb = keras.datasets.imdb
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=10000)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
word_to_index = imdb.get_word_index()
word_to_index = {k: (v + 3) for k, v in word_to_index.items()}
word_to_index["<PAD>"] = 0    # 패딩(길이 맞추기) 기호
word_to_index["<START>"] = 1  # 시작 기호
word_to_index["<UNK>"] = 2    # 알 수 없는 토큰 (Unknown)
word_to_index["<UNUSED>"] = 3 # 사용되지 않는 토큰

index_to_word = dict([(value, key) for (key, value) in word_to_index.items()])

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
x_train = pad_sequences(x_train, maxlen=100)
x_test = pad_sequences(x_test, maxlen=100)

In [5]:
vocab_size = 10000

model = Sequential()
model.add(Embedding(vocab_size, 64, input_length=100))
model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5)) # 과적합 방지를 위한 드롭아웃
model.add(Dense(1, activation='sigmoid')) # 이진 분류이므로 시그모이드 사용

print("\n[모델 구조]")
model.summary()


[모델 구조]


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [6]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

print("\n[모델 학습 시작]")
history = model.fit(x_train, y_train,
                    batch_size=64, epochs=20, verbose=1,
                    validation_data=(x_test, y_test))


[모델 학습 시작]
Epoch 1/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 12s 27ms/step - accuracy: 0.7643 - loss: 0.4637 - val_accuracy: 0.8496 - val_loss: 0.3407
Epoch 2/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - accuracy: 0.9396 - loss: 0.1715 - val_accuracy: 0.8332 - val_loss: 0.4051
Epoch 3/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 9s 24ms/step - accuracy: 0.9933 - loss: 0.0296 - val_accuracy: 0.8228 - val_loss: 0.5596
Epoch 4/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - accuracy: 0.9993 - loss: 0.0061 - val_accuracy: 0.8350 - val_loss: 0.6222
Epoch 5/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 11s 27ms/step - accuracy: 0.9999 - loss: 0.0022 - val_accuracy: 0.8340 - val_loss: 0.6716
Epoch 6/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - accuracy: 1.0000 - loss: 9.9280e-04 - val_accuracy: 0.8366 - val_loss: 0.7039
Epoch 7/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 11s 29ms/step - accuracy: 1.0000 - loss: 5.5922e-04 - val_accuracy: 0.8374 - val_loss: 0.7389
Epoch 8/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 10s 26ms/step - accuracy: 1.000

In [7]:
print("\n[모델 평가]")
results = model.evaluate(x_test, y_test, verbose=2)
print(f"테스트 손실(Loss): {results[0]:.4f}, 테스트 정확도(Accuracy): {results[1]:.4f}")


[모델 평가]
782/782 - 2s - 3ms/step - accuracy: 0.8226 - loss: 1.2804
테스트 손실(Loss): 1.2804, 테스트 정확도(Accuracy): 0.8226


In [10]:
import re


review = """What can I say about this movie that was already said? It is my favorite time travel sci-fi, adventure epic comedy in the 80's and I love this movie to death!
When I saw this movie I was thrown out by its theme. An excellent sci-fi, adventure epic, I LOVE the 80s.
It's simple the greatest time travel movie ever happened in the history of world cinema. I love this movie to death, I love, LOVE, love it!"""

review = re.sub("[^0-9a-zA-Z ]", "", review).lower()

review_encoding = []
for w in review.split():

    index = word_to_index.get(w, 2)
    if index <= 10000:
        review_encoding.append(index)
    else:
        review_encoding.append(word_to_index["<UNK>"])

In [11]:
test_input = pad_sequences([review_encoding], maxlen=100)
value = model.predict(test_input)

print("\n[새로운 리뷰 예측 결과]")
if (value > 0.5):
    print(f"예측 확률 {value[0][0]:.4f} -> 긍정적인 리뷰입니다.")
else:
    print(f"예측 확률 {value[0][0]:.4f} -> 부정적인 리뷰입니다.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step

[새로운 리뷰 예측 결과]
예측 확률 1.0000 -> 긍정적인 리뷰입니다.
